# Capstone 1 — SEC Filings RAG (compressed, ~2.5h)

Spec: `README.md` in this folder. Build a notebook that ingests 2–3 10-Ks, indexes in Chroma, and answers 5 sample analyst questions with chunk-level citations.

LangChain cheatsheet: `../../langchain_langgraph.ipynb` §§2, 4, 8, 12.

**Time blocks**
| Block | Section | Goal |
|---|---|---|
| 0:00–0:30 | H1 | Corpus pull + clean |
| 0:30–1:00 | H2 | Chunk + embed + persist Chroma |
| 1:00–1:30 | H3 | Retriever-as-tool |
| 1:30–2:00 | H4 | Citation-grounded agent |
| 2:00–2:30 | H5 | 5 sample questions + LangSmith traces |

## Setup

Run this once at the start of the kernel.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# load .env from this folder, the capstones folder, the llm-pipeline folder, or the repo root — whichever has it
load_dotenv()

HERE = Path.cwd()
DATA = HERE / "data"
FILINGS_DIR = DATA / "filings"
CHROMA_DIR = DATA / "chroma"

os.environ.setdefault("LANGSMITH_PROJECT", "capstone-1-rag")

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY missing — check .env"
assert ".venv" in sys.executable, f"Wrong kernel: {sys.executable}. Switch to .venv/bin/python."
print("python    :", sys.executable)
print("filings   :", FILINGS_DIR, "exists:", FILINGS_DIR.exists())
print("chroma    :", CHROMA_DIR, "exists:", CHROMA_DIR.exists())
print("langsmith :", os.environ.get("LANGSMITH_TRACING", "off"))

---
## H1 — Corpus pull (0:00–0:30)

Goal: pull JPM + BAC + GS most-recent 10-K from EDGAR; clean HTML → text; persist to `data/filings/{TICKER}_10K.txt`.

**Hints**
- `from sec_edgar_downloader import Downloader` — mandatory `email_address` (User-Agent).
- Raw output lands at `data/sec-edgar-filings/<TICKER>/10-K/<accession>/full-submission.txt`.
- Cleaning: `BeautifulSoup(raw, 'lxml').get_text(separator='\n')`, then collapse `\n\n\n+` → `\n\n`.
- **Idempotent**: skip download if `data/sec-edgar-filings/<TICKER>` already populated; skip cleaning if cleaned `.txt` exists.
- **Sanity**: cleaned files should be 150k–400k chars. <50k = cleaner ate body. >1M = exhibits not stripped.

In [ ]:
# from sec_edgar_downloader import Downloader
# from bs4 import BeautifulSoup
# import re

TICKERS = ["JPM", "BAC", "GS"]

# --- TODO: download via sec_edgar_downloader ---
# dl = Downloader(
#     company_name="<your name>",
#     email_address="<your email>",
#     download_folder=str(DATA),
# )
# for ticker in TICKERS:
#     dl.get("10-K", ticker, limit=1)

# --- TODO: clean raw submission → plain text → FILINGS_DIR/{ticker}_10K.txt ---
# def clean_filing(raw_path: Path) -> str:
#     ...
#
# for ticker in TICKERS:
#     out = FILINGS_DIR / f"{ticker}_10K.txt"
#     if out.exists():
#         continue
#     ...  # find raw, clean, write

# Your code here:


In [ ]:
# Sanity — char counts per cleaned filing
for ticker in TICKERS:
    p = FILINGS_DIR / f"{ticker}_10K.txt"
    if p.exists():
        n = len(p.read_text())
        flag = "✓" if 150_000 <= n <= 800_000 else "⚠ check cleaner"
        print(f"{ticker}: {n:>9,d} chars  {flag}")
    else:
        print(f"{ticker}: ⚠ missing")

---
## H2 — Chunk + embed (0:30–1:00)

Goal: chunk each filing, embed with `text-embedding-3-small`, persist to Chroma.

**Hints**
- `RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)`.
- Each `Document.metadata` MUST carry `{ticker, chunk_idx, source}` — this becomes the citation token.
- **Idempotent**: if `CHROMA_DIR` is non-empty, load (`Chroma(persist_directory=..., embedding_function=...)`); else build + persist with `Chroma.from_documents(...)`.
- **Sanity**: total chunks should be ~300–800.

In [ ]:
# from langchain_community.vectorstores import Chroma
# from langchain_openai import OpenAIEmbeddings
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain_core.documents import Document

# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)

# def build_or_load_vector_store():
#     if CHROMA_DIR.exists() and any(CHROMA_DIR.iterdir()):
#         return Chroma(persist_directory=str(CHROMA_DIR), embedding_function=embeddings)
#     docs: list[Document] = []
#     for ticker in TICKERS:
#         text = (FILINGS_DIR / f"{ticker}_10K.txt").read_text()
#         for i, chunk in enumerate(splitter.split_text(text)):
#             docs.append(Document(
#                 page_content=chunk,
#                 metadata={"ticker": ticker, "chunk_idx": i, "source": f"{ticker}_10K.txt"},
#             ))
#     return Chroma.from_documents(docs, embeddings, persist_directory=str(CHROMA_DIR))

vector_store = ...  # your implementation

In [ ]:
# Sanity — collection size
n = vector_store._collection.count()
flag = "✓" if 300 <= n <= 1500 else "⚠ check chunk_size"
print(f"chunks: {n}  {flag}")

---
## H3 — Retriever-as-tool (1:00–1:30)

Goal: a `@tool` that retrieves top-k chunks and returns them as a string with `[TICKER#idx]` tokens, ready to cite.

**Hints**
- Cheatsheet §8. Wrap the retriever — the agent decides *when* to call it.
- Return a string, NOT `Document` objects. Format each result as `[{ticker}#{chunk_idx}] {page_content}`, joined by `\n\n---\n\n`.
- `vector_store.as_retriever(search_kwargs={'k': 4})`.

In [ ]:
# from langchain_core.tools import tool

# retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# @tool
# def search_filings(query: str) -> str:
#     """Search the SEC 10-K knowledge base for relevant passages.
#     Use this for any factual question about JPM, BAC, or GS — capital ratios,
#     trading-book VaR, litigation risks, climate disclosures, etc.
#     Returns up to 4 chunks formatted as [TICKER#idx] <chunk text>, separated by ---.
#     """
#     # TODO: invoke retriever, format each result, join
#     ...

search_filings = ...  # your @tool

In [ ]:
# Sanity — retrieval works on a known-easy query
print(search_filings.invoke("tier 1 capital ratio")[:1500])

---
## H4 — Citation-grounded answer (1:30–2:00)

Goal: an agent that calls the retriever, returns `{answer, citations}`, and **only** cites IDs that appeared in retrieved context.

**Hints**
- Pydantic schema: `CitedAnswer(answer: str, citations: list[str])`.
- The system prompt MUST contain: *"You may only cite chunk IDs that appeared in retrieved context. If retrieval is insufficient, say so explicitly and return an empty citations list."* Otherwise the LLM hallucinates IDs.
- `init_chat_model('openai:gpt-4o-mini')` (cheap; cheatsheet §2).
- `create_react_agent(llm, [search_filings], prompt=SYSTEM_PROMPT)` (cheatsheet §1, §5).
- For structured-output binding inside the agent loop, simplest path is to instruct the model to format its final answer as JSON matching `CitedAnswer` and parse it post-hoc; or use `with_structured_output` outside the agent and orchestrate retrieval manually. Pick one approach and stick to it.

In [ ]:
# from pydantic import BaseModel, Field
# from langchain.chat_models import init_chat_model
# from langgraph.prebuilt import create_react_agent

# class CitedAnswer(BaseModel):
#     answer: str = Field(description="The answer text. Cite chunk IDs inline like [JPM#42].")
#     citations: list[str] = Field(description="Chunk IDs cited; each must appear in retrieved context.")

# SYSTEM_PROMPT = """You are an analyst's research assistant working over SEC 10-K filings
# for JPM, BAC, and GS.
# Rules:
#   1. Always call `search_filings` before answering a factual question.
#   2. You may ONLY cite chunk IDs that appeared in retrieved context. Never fabricate IDs.
#   3. If retrieval is insufficient to answer, say so explicitly and return an empty citations list.
#   4. Cite inline like [TICKER#idx]. Be concise; analysts hate filler.
# """

# llm = init_chat_model("openai:gpt-4o-mini", temperature=0)
# agent = create_react_agent(llm, [search_filings], prompt=SYSTEM_PROMPT)

agent = ...  # your agent

---
## H5 — 5 sample questions (2:00–2:30)

Run all 5 with LangSmith tracing on. Q4 (cross-doc compare) is *expected* to be weak — that's the talking point about query decomposition as production work.

In [ ]:
QUESTIONS = [
    "What is each company's tier 1 capital ratio as of the most recent fiscal year?",
    "Which company has the highest exposure to commercial real estate?",
    "Summarise the litigation risks disclosed by JPM.",
    "Compare the trading-book VaR methodology between BAC and GS.",
    "What does GS say about climate-related financial risk?",
]

# for i, q in enumerate(QUESTIONS):
#     config = {"tags": ["capstone-1"], "metadata": {"q_idx": i}}
#     result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=config)
#     final = result["messages"][-1].content
#     print(f"\n[Q{i}] {q}\n{'-' * 80}\n{final[:1200]}\n")

---
## End-of-day notes

Fill in once you've run the 5 questions. ~1 paragraph.

- **Worked**: …
- **Failed** (expected: Q4): …
- **Three LangSmith trace links**:
  1. …
  2. …
  3. …
- **What I'd add for production**: 30-Q hand-authored eval set with hit-rate@5 + faithfulness LLM-judge (stronger model than the agent), three-column baseline (no-retrieval / dense-only / hybrid+rerank), FastAPI streaming endpoint with `MemorySaver`, prompt caching on the system prompt. (See `README.md` → "What got cut".)